# Leakage audit

**Goal:** find every way the old dataset lets the model cheat, prove it with experiments, and write down what a *fair* dataset needs.

Read `docs/leakage_audit.md` alongside this notebook.

The four kinds of leakage we look for:
1. **Target leakage:** a feature secretly contains the label.
2. **Group leakage:** train and test share the same images/nights, so the model memorises them.
3. **Shortcut learning:** the classes differ in an easy, unrealistic way.
4. **Distribution shift:** the test data doesn't look like the real data the model will face.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from asthunt.config.load_config import PROJECT_ROOT
from asthunt.evaluate.check_split_leakage import check_split_leakage
from asthunt.link.fit_linear_motion import fit_linear_motion
from asthunt.models.train_baseline_classifier import train_baseline_classifier

tracklets = pd.read_csv(PROJECT_ROOT / "legacy" / "tracklets.csv")
rng = np.random.default_rng(42)

## 1. Target leakage
Already found in *ML basics*: `mag`, `true_rate` and `true_angle` are filled only for injected (real) rows.
Rule: **only features you could measure on unlabelled sky data.**

In [ ]:
for column in ["mag", "true_rate", "true_angle"]:
    missing = tracklets.groupby("label")[column].apply(lambda c: c.isna().mean())
    print(f"{column:11s} missing in bogus: {missing[0]:.0%}   missing in real: {missing[1]:.0%}")

## 2. Group leakage
`legacy/generate_dataset.py` made **every** tracklet from the same 4 images (`FITS_FILES[:4]`).
So every row belongs to one group, and any split puts that group in both train and test.

In [ ]:
tracklets["image_set"] = "the same 4 images"     # what generate_dataset.py used for every row

train_rows, test_rows = train_test_split(tracklets, test_size=0.25, random_state=42)
print("groups in both train and test:", check_split_leakage(train_rows["image_set"], test_rows["image_set"]))

The model is tested on the same sky, the same stars and the same artifacts it trained on.
We cannot fix this with this file: we need **many nights**, then split by night (the *Dataset building* phase).

## 3. Shortcut learning: the bogus rows are too easy
Bogus rows were 4 random sources from anywhere on the chip. How would a *real* bogus tracklet look?
The most common one: the linker joins 3 detections of a real asteroid plus **one wrong source a few pixels away**
(a noise spike, a cosmic ray, a faint star). We call that a **near miss**.

We make near misses by copying each real tracklet and moving one of its 4 detections by 3 to 10 pixels.

In [ ]:
TIMES_HOURS = [0.0, 0.25, 0.50, 0.75]    # the time steps the legacy code assumed
PIXEL_SCALE = 1.012                      # arcsec per pixel (from the FITS header)


def describe_track(x_positions, y_positions):
    """Return the two legacy features for one track: rate ("/hr) and max residual (px)."""
    fit = fit_linear_motion(TIMES_HOURS, x_positions, y_positions)
    rate = np.hypot(fit["vx"], fit["vy"]) * PIXEL_SCALE
    return rate, fit["max_residual"]


def make_near_miss(row):
    """Copy a real tracklet and push one random detection 3-10 px in a random direction."""
    xs = [row["x1"], row["x2"], row["x3"], row["x4"]]
    ys = [row["y1"], row["y2"], row["y3"], row["y4"]]
    which = rng.integers(0, 4)
    distance = rng.uniform(3, 10)
    angle = rng.uniform(0, 2 * np.pi)
    xs[which] += distance * np.cos(angle)
    ys[which] += distance * np.sin(angle)
    rate, residual = describe_track(xs, ys)
    return {"measured_rate": rate, "linearity_residual": residual, "label": 0}


real_rows = tracklets[tracklets["label"] == 1]
near_misses = pd.DataFrame([make_near_miss(row) for _, row in real_rows.iterrows()])
print(len(near_misses), "near-miss bogus tracklets")
near_misses.describe().round(2)

In [ ]:
easy_bogus = tracklets[tracklets["label"] == 0]

bins = np.linspace(-2.5, 3.5, 70)
plt.hist(np.log10(real_rows["linearity_residual"]), bins=bins, alpha=0.6, label="real")
plt.hist(np.log10(near_misses["linearity_residual"]), bins=bins, alpha=0.6, label="near-miss bogus (realistic)")
plt.hist(np.log10(easy_bogus["linearity_residual"]), bins=bins, alpha=0.6, label="random bogus (legacy)")
plt.xlabel("log10(linearity residual, px)"); plt.legend(); plt.show()

Near misses overlap with real tracklets. Now the problem is actually hard, like on the real sky.

## 4. Distribution shift: train on easy, test on realistic
We train on the legacy data, then test on **real tracklets + near misses** the model never saw.

In [ ]:
feature_columns = ["measured_rate", "linearity_residual"]

def log_features(table):
    # log10 makes these very skewed numbers friendlier for a linear model (see ML basics, exercise 1)
    return np.log10(table[feature_columns].to_numpy())

real_train, real_test = train_test_split(real_rows, test_size=0.25, random_state=42)
easy_train, easy_test = train_test_split(easy_bogus, test_size=0.25, random_state=42)

# Near misses were made FROM real tracklets. Keep each pair on the same side of the split,
# otherwise a near miss in test has its almost-identical real twin in train (leakage again!).
near_train = near_misses.loc[real_rows.index.get_indexer(real_train.index)]
near_test = near_misses.loc[real_rows.index.get_indexer(real_test.index)]

easy_training_set = pd.concat([real_train, easy_train])
model_trained_on_easy = train_baseline_classifier(log_features(easy_training_set), easy_training_set["label"])

easy_test_set = pd.concat([real_test, easy_test])
realistic_test_set = pd.concat([real_test, near_test])
print("trained on easy, tested on easy     :", round(model_trained_on_easy.score(log_features(easy_test_set), easy_test_set["label"]), 3))
print("trained on easy, tested on realistic:", round(model_trained_on_easy.score(log_features(realistic_test_set), realistic_test_set["label"]), 3))

The same model, the same real asteroids. Only the bogus examples got realistic, and the score collapses.
**The legacy test score measured the dataset, not the model.**

What if we train on realistic bogus too?

In [ ]:
realistic_training_set = pd.concat([real_train, near_train])
model_trained_on_realistic = train_baseline_classifier(log_features(realistic_training_set), realistic_training_set["label"])
print("trained on realistic, tested on realistic:",
      round(model_trained_on_realistic.score(log_features(realistic_test_set), realistic_test_set["label"]), 3))

Back to 99%. Two lessons:
1. **The model has to see hard cases to learn them.** Bogus examples chosen because they look like the real thing are called
   **hard negatives**, and they're the most valuable part of a training set.
2. **99% is still optimistic.** Offsets of 3-10 px are the *easy* end of near misses. Exercise 1 makes the offset 1-3 px
   (accuracy falls to about 96%) and 0.5-1.5 px gives about 89%. A wrong source 1 px off a line is almost invisible to position features.
   Real bogus also includes artifacts (cosmic rays, bad subtractions) that sit on a perfectly straight line by chance.

To do better we need **more information**: does the brightness stay constant? Does each detection *look* like a star-shaped blob
(the CNN's job)? That's exactly what the next phases build.

## 5. What a fair dataset needs (the spec we carry forward)
| Problem found | Rule for the new dataset |
|---|---|
| `mag`, `true_*` filled only for real rows | Features = only things measured from detections |
| All rows from the same 4 images | Many nights; split **by night**; check with `check_split_leakage` |
| Random bogus = trivially easy | Bogus = real detections linked by the linker (near misses, artifacts, chance alignments) |
| Fake 15-minute spacing between frames from different months | Use frames from the **same night** with their real timestamps |
| Near-miss twins could straddle the split | Derived examples stay in the same split as their source |

## Exercises
1. Change the near-miss offset to 1-3 px. What happens to the realistic accuracy? Why is that the hardest case?
2. Remove the "keep each pair on the same side" step and use the plain `near_train, near_test` split. Does the score go up? Why is that number dishonest?
3. Think of one more kind of realistic bogus tracklet. How would you generate it?